In [3]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load all datasets
print("Loading datasets...")

race_df = pd.read_csv('data/processed/race_results_2025-6_clean.csv')
reddit_df = pd.read_csv('data/raw/reddit_posts.csv')
youtube_df = pd.read_csv('data/processed/youtube_engagement_2025-6.csv')
news_df = pd.read_csv('data/processed/news_mentions.csv')

print(f"Race Results: {len(race_df)} rows")
print(f"Reddit Mentions: {len(reddit_df)} rows")
print(f"YouTube Engagement: {len(youtube_df)} rows")
print(f"News Mentions: {len(news_df)} rows")

Loading datasets...
Race Results: 2013 rows
Reddit Mentions: 77 rows
YouTube Engagement: 5 rows
News Mentions: 5 rows


In [4]:
def audit_dataset(df, name):
    """
    Print comprehensive audit of dataset structure.
    """
    print(f"\n{'='*50}")
    print(f"DATASET: {name}")
    print(f"{'='*50}")
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    print(f"\nColumn Names and Types:")
    print("-" * 40)
    for col in df.columns:
        dtype = df[col].dtype
        sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else "N/A"
        print(f"  {col:25} | {str(dtype):10} | Sample: {str(sample)[:30]}")
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print("  None")
    return None

# Audit each dataset
audit_dataset(race_df, "Race Results")
audit_dataset(reddit_df, "Reddit Mentions")
audit_dataset(youtube_df, "YouTube Engagement")
audit_dataset(news_df, "News Mentions")


DATASET: Race Results
Shape: 2013 rows x 33 columns

Column Names and Types:
----------------------------------------
  year                      | int64      | Sample: 2025
  race_number               | int64      | Sample: 1
  race_name                 | str        | Sample: 2025 Daytona 500
  Race_Date                 | str        | Sample: 2025-02-16
  track                     | str        | Sample: Daytona International Speedway
  track_type                | str        | Sample: road course
  track_miles               | float64    | Sample: 2.4
  total_laps                | float64    | Sample: 95.0
  caution_flags             | int64      | Sample: 8
  caution_laps              | int64      | Sample: 47
  lead_changes              | int64      | Sample: 56
  avg_speed_mph             | float64    | Sample: 129.159
  pole_speed_mph            | float64    | Sample: 182.745
  margin_of_victory         | str        | Sample: .113 sec
  attendance                | float64    | Samp

In [5]:
# Define standard date format
DATE_FORMAT = '%Y-%m-%d'  # ISO 8601: 2024-02-18

def standardize_dates(df, date_columns):
    """
    Convert all date columns to standard format.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    date_columns : list
        List of column names containing dates

    Returns:
    --------
    pd.DataFrame : DataFrame with standardized dates
    """
    df = df.copy()

    for col in date_columns:
        if col not in df.columns:
            print(f"  Warning: Column '{col}' not found")
            continue

        # Convert to datetime
        df[col] = pd.to_datetime(df[col], errors='coerce')

        # Check for conversion failures
        failed = df[col].isna().sum()
        if failed > 0:
            print(f"  Warning: {failed} dates failed to convert in '{col}'")

        # Convert to standard string format for CSV compatibility
        df[f'{col}_str'] = df[col].dt.strftime(DATE_FORMAT)

    return df

# Apply to each dataset
print("Standardizing dates...")

race_df = standardize_dates(race_df, ['Race_Date'])
reddit_df = standardize_dates(reddit_df, ['race_date'] if 'race_date' in reddit_df.columns else [])
youtube_df = standardize_dates(youtube_df, ['race_date'])
news_df = standardize_dates(news_df, ['race_date'] if 'race_date' in news_df.columns else [])

print("Date standardization complete.")

Standardizing dates...
Date standardization complete.


In [6]:
def standardize_columns(df, column_mapping):
    """
    Rename columns to standard names and convert to lowercase with underscores.

    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame to standardize
    column_mapping : dict
        Dictionary mapping old names to new names

    Returns:
    --------
    pd.DataFrame : DataFrame with standardized column names
    """
    df = df.copy()

    # Apply explicit mappings
    df = df.rename(columns=column_mapping)

    # Convert remaining columns to lowercase with underscores
    df.columns = df.columns.str.lower().str.replace(' ', '_')

    return df

# Define standard column names for each dataset
race_columns = {
    'Race_Name': 'race_name',
    'Race_Date': 'race_date',
    'Race_Number': 'race_number',
    'Driver': 'driver',
    'Team': 'team',
    'Sponsor': 'sponsor',
    'Finish_Position': 'finish_position',
    'Laps_Led': 'laps_led'
}

exposure_columns = {
    'race_period': 'race_name',  # If Reddit uses race_period
    'Race_Name': 'race_name',
    'Race_Number': 'race_number',
    'Sponsor': 'sponsor'
}

# Apply standardization
race_df = standardize_columns(race_df, race_columns)
reddit_df = standardize_columns(reddit_df, exposure_columns)
youtube_df = standardize_columns(youtube_df, exposure_columns)
news_df = standardize_columns(news_df, exposure_columns)

print("Column name standardization complete.")

Column name standardization complete.


In [ ]:
# add race numbers based on date:
date_to_number = {pd.date_range(start='2025-02-0', end='2025-2-11'): 1,
                  pd.date_range(start='2025-2-12', end='2025-2-21'): 2,
                  pd.date_range(start='2025-2-22', end='2025-2-28'): 3,
                  pd.date_range(start='2025-2-29', end='2025-3-7'): 4,
                  pd.date_range(start='2025-3-8', end='2025-3-14'): 5,
                  pd.date_range(start='2025-3-15', end='2025-3-21'): 6,
                  pd.date_range(start='2025-3-22', end='2025-3-29'): 7,
                  pd.date_range(start='2025-3-30', end='2025-4-4'): 8,
                  pd.date_range(start='2025-4-5', end='2025-4-11'): 9,
                  pd.date_range(start='2025-4-12', end='2025-4-25'): 10,
                  pd.date_range(start='2025-4-26', end='2025-5-2'): 11,
                  pd.date_range(start='2025-5-3', end='2025-5-9'): 12,
                  pd.date_range(start='2025-5-10', end='2025-5-16'): 13,
                  pd.date_range(start='2025-5-17', end='2025-5-23'): 14,
                  pd.date_range(start='2025-5-24', end='2025-5-30'): 15,
                  pd.date_range(start='2025-5-31', end='2025-6-6'): 16,
                  pd.date_range(start='2025-6-7', end='2025-6-13'): 17,
                  pd.date_range(start='2025-6-14', end='2025-6-20'): 18,
                  pd.date_range(start='2025-6-21', end='2025-6-26'): 19,
                  pd.date_range(start='2025-6-27', end='2025-7-4'): 20,
                  pd.date_range(start='2025-7-5', end='2025-7-11'): 21,
                  pd.date_range(start='2025-7-12', end='2025-7-18'): 22,
                  pd.date_range(start='2025-7-19', end='2025-7-25'): 23,
                  pd.date_range(start='2025-7-26', end='2025-8-1'): 24,
                  pd.date_range(start='2025-8-2', end='2025-8-8'): 25,
                  pd.date_range(start='2025-8-9', end='2025-8-14'): 26,
                  pd.date_range(start='2025-8-15', end='2025-8-21'): 27,
                  pd.date_range(start='2025-8-22', end='2025-8-29'): 28,
                  pd.date_range(start='2025-8-30', end='2025-9-5'): 29,
                  pd.date_range(start='2025-9-6', end='2025-9-11'): 30,
                  pd.date_range(start='2025-9-12', end='2025-9-19'): 31,
                  pd.date_range(start='2025-9-20', end='2025-9-26'): 32,
                  pd.date_range(start='2025-9-27', end='2025-10-3'): 33,
                  pd.date_range(start='2025-10-4', end='2025-10-11'): 34,
                  pd.date_range(start='2025-10-12', end='2025-10-17'): 35,
                  pd.date_range(start='2025-10-18', end='2025-10-26'): 36,
                  pd.date_range(start='2025-10-27', end='2025-10-31'): 37,
                  pd.date_range(start='2025-11-1', end='2026-2-20'): 38,
                  pd.date_range(start='2026-2-21', end='2026-2-27'): 39,
                  pd.date_range(start='2026-2-28', end='2026-3-6'): 40,
                  pd.date_range(start='2026-3-7', end='2026-3-13'): 41,
                  pd.date_range(start='2026-3-14', end='2026-3-20'): 42,
                  pd.date_range(start='2026-3-21', end='2026-3-27'): 43,
                  pd.date_range(start='2026-3-28', end='2026-4-10'): 44,
                  pd.date_range(start='2026-4-11', end='2026-4-17'): 45,
                  pd.date_range(start='2026-4-18', end='2026-4-24'): 46,
                  pd.date_range(start='2026-4-25', end='2026-5-1'): 47,
                  pd.date_range(start='2026-5-2', end='2026-5-8'): 48,
                  pd.date_range(start='2026-5-9', end='2026-5-15'): 49,
                  pd.date_range(start='2026-5-16', end='2026-5-22'): 50,
                  pd.date_range(start='2026-5-23', end='2026-5-29'): 51,
                  pd.date_range(start='2026-5-30', end='2026-6-5'): 52,
                  pd.date_range(start='2026-6-6', end='2026-6-12'): 53,
}

def date_to_race(df):
    """
    Convert a date to its corresponding race number based on the predefined date ranges.
    """
    
                  
                  
                    

In [7]:
def create_race_name_mapping(df, race_col='race_name'):
    """
    Create a standardized mapping for race names.
    Returns dict mapping original names to standardized names.
    """
    unique_races = df[race_col].unique()
    print(f"Found {len(unique_races)} unique race names:")
    for race in sorted(unique_races):
        print(f"  - {race}")
    return unique_races

# Check race names in each dataset
print("\n=== Race Names in Race Results ===")
race_names_main = create_race_name_mapping(race_df)

print("\n=== Race Names in Reddit Data ===")
race_names_reddit = create_race_name_mapping(reddit_df)

print("\n=== Race Names in YouTube Data ===")
race_names_youtube = create_race_name_mapping(youtube_df)

print("\n=== Race Names in News Data ===")
race_names_news = create_race_name_mapping(news_df)


=== Race Names in Race Results ===
Found 52 unique race names:
  - 2025 AdventHealth 400
  - 2025 Ambetter Health 400
  - 2025 Autotrader EchoPark Automotive 400
  - 2025 Bank of America ROVAL 400
  - 2025 Bass Pro Shops Night Race
  - 2025 Brickyard 400 Presented by PPG
  - 2025 Coca-Cola 600
  - 2025 Coke Zero Sugar 400
  - 2025 Cook Out 400
  - 2025 Cracker Barrel 400
  - 2025 Cup Series Championship
  - 2025 Daytona 500
  - 2025 EchoPark Automotive Grand Prix
  - 2025 Enjoy Illinois 300
  - 2025 Firekeepers Casino 400
  - 2025 Food City 500
  - 2025 Go Bowling at The Glen
  - 2025 Goodyear 400
  - 2025 Grant Park 165
  - 2025 Hollywood Casino 400
  - 2025 Iowa Corn 350
  - 2025 Jack Links 500
  - 2025 Mobil 1 301
  - 2025 Pennzoil 400
  - 2025 Quaker State 400 available at Walmart
  - 2025 Shriners Childrens 500
  - 2025 South Point 400
  - 2025 Southern 500
  - 2025 Straight Talk Wireless 400
  - 2025 The Great American Getaway 400
  - 2025 Toyota / Save Mart 350
  - 2025 Viva Me

KeyError: 'race_name'